# QCDark2 Validation

Visual comparison against upstream QCDark2 dielectric-function spectra.

In [ ]:
import os
import sys
import warnings

sys.path.insert(0, '..')
sys.path.insert(0, '.')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')

import numpy as np
import matplotlib.pyplot as plt
import numericalunits as nu
from scipy.integrate import simpson

nu.reset_units('SI')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='torchquad')
plt.rcParams['text.usetex'] = False
plt.rcParams['figure.dpi'] = 120

BENCHMARK_MASSES_MEV = [10.0, 100.0, 1000.0]
BENCHMARK_FDM = {'heavy': 0, 'light': 2}
DEFAULT_NE_BINS = [1, 2, 3, 4, 5]

from tests.conftest import qcdark2_file, upstream_qcdark2_file, UPSTREAM_QCDARK2_ROOT
from DMeRates.DMeRate import DMeRate

sys.path.insert(0, UPSTREAM_QCDARK2_ROOT)
from qcdark2 import dark_matter_rates as dm2
plt.rcParams['text.usetex'] = False

ALL_QCDARK2_CASES = [
    ('Si', 'composite'), ('Si', 'lfe'), ('Si', 'nolfe'),
    ('Ge', 'composite'), ('Ge', 'lfe'), ('Ge', 'nolfe'),
    ('GaAs', 'composite'), ('GaAs', 'lfe'), ('GaAs', 'nolfe'),
    ('SiC', 'composite'), ('SiC', 'lfe'), ('SiC', 'nolfe'),
    ('Diamond', 'composite'), ('Diamond', 'lfe'), ('Diamond', 'nolfe'),
]

In [ ]:
def _ratio(candidate, reference):
    ratio = np.full_like(reference, np.nan, dtype=float)
    mask = np.isfinite(reference) & (reference > np.max(reference) * 1e-10)
    ratio[mask] = candidate[mask] / reference[mask]
    return ratio, mask


def plot_spectrum_overlay(energy, dmrates, reference, title, reference_label='upstream'):
    ratio, mask = _ratio(dmrates, reference)
    fig, (ax, rx) = plt.subplots(2, 1, figsize=(7, 5), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
    ax.plot(energy, dmrates, lw=2, label='DMeRates')
    ax.plot(energy, reference, '--', lw=2, label=reference_label)
    ax.set_yscale('log')
    ax.set_ylabel('events/kg/year/eV')
    ax.set_title(title)
    ax.legend()
    rx.plot(energy, ratio, color='black', lw=1.5)
    rx.fill_between(energy, 0.95, 1.05, color='tab:green', alpha=0.18)
    rx.axhline(1.0, color='0.4', lw=1)
    rx.set_ylim(0.5, 1.5)
    rx.set_xlabel('energy [eV]')
    rx.set_ylabel('ratio')
    plt.show()
    if np.any(mask):
        total_dm = simpson(dmrates, x=energy)
        total_ref = simpson(reference, x=energy)
        max_resid = np.nanmax(np.abs(ratio[mask] - 1.0))
        print(f'total rate (DMeRates): {total_dm:.6e} events/kg/year')
        print(f'total rate (upstream): {total_ref:.6e} events/kg/year')
        print(f'rel. difference: {(total_dm / total_ref - 1.0):+.3%}')
        print(f'max bin residual: {max_resid:.3%}')

In [ ]:
def compare_material_variant(material, variant, mass_mev=100.0, fdm_n=0):
    mediator = 'heavy' if fdm_n == 0 else 'light'
    epsilon = dm2.load_epsilon(upstream_qcdark2_file(material, variant))
    reference, energy_ref = dm2.get_dR_dE(
        epsilon,
        m_X=mass_mev * 1e6,
        mediator=mediator,
        astro_model=dm2.default_astro,
        screening='RPA',
    )
    obj = DMeRate(material, form_factor_type='qcdark2', qcdark2_variant=variant, device='cpu')
    obj.update_params(dm2.default_astro['v0'], dm2.default_astro['vEarth'], dm2.default_astro['vEscape'], dm2.default_astro['rhoX'], dm2.default_astro['sigma_e'])
    energy_dm, spectra_dm = obj.calculate_spectrum([mass_mev], 'imb', fdm_n, DoScreen=False)
    assert np.allclose(energy_dm, energy_ref)
    plot_spectrum_overlay(energy_dm, spectra_dm[0], reference, f'QCDark2 {material}/{variant}, {mediator}')
    return obj, energy_dm, spectra_dm[0], reference

for material, variant in ALL_QCDARK2_CASES:
    compare_material_variant(material, variant, 100.0, 0)

In [ ]:
obj = DMeRate('Si', form_factor_type='qcdark2', qcdark2_variant='composite', device='cpu')
obj.update_params(238.0, 250.2, 544.0, 0.3e9, 1e-39)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    screen = obj.calculate_total_rate([100.0], 'imb', 0, DoScreen=True)
no_screen = obj.calculate_total_rate([100.0], 'imb', 0, DoScreen=False)
print('DoScreen=True:', screen)
print('DoScreen=False:', no_screen)
print('equal:', np.array_equal(screen, no_screen))
print('warning:', caught[0].message if caught else 'none')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for variant in ['nolfe', 'lfe', 'composite']:
    obj, energy, spectrum, _ = compare_material_variant('Si', variant, 100.0, 0)
    ax.plot(energy, spectrum, label=variant)
ax.set_yscale('log')
ax.set_xlabel('energy [eV]')
ax.set_ylabel('events/kg/year/eV')
ax.set_title('Si QCDark2 variant comparison')
ax.legend()
plt.show()